<a href="https://colab.research.google.com/github/Nesrine-larbi/COMP333-Project/blob/main/teamd_phase1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Course: COMP 333 — Project Phase 1: Data Acquisition & Baseline

**Team D**:
- Ronnie Chan (27206003)
- Patrice Gallant (40301020)
- Nesrine Larbi (40079009)

<br><br>

### **Source of the Datasets**
https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

<br><br>

### **Division-of-Labour Statement**
- **Ronnie Chan**: Data Retrieval, Python script `taxi_utils.py`, Data Wrangling/Cleaning
- **Patrice Gallant**: Data Wrangling/Cleaning, Baseline model
- **Nesrine Larbi**: Data Wrangling/Cleaning, EDA

## 1.Import Libraries

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import gc
from taxi_utils import TaxiData, TaxiDDA        # Python script for downloading data and DDA

## 2.Data Retrieval

- **Objective**: This section covers the retrieval of our dataset from the [NYC Taxi and Limousine Commission (TLC)](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) website.

- **Data Selection:** We selected data from June and July 2025 to represent the Peak Summer Tourism Season 2025. The data is provided in PARQUET format, a columnar storage that allows us to handle millions of raw entries efficiently through compression.

- **Programmatic Retrieval:** The downloading process is implemented in `taxi_utils.py` using the `curl` command. The `TaxiData` class automates the acquisition of these specific months to ensure reproducibility. Given the 12.7 GB RAM constraint of our environment (Google Colab), we built a dataset with a manageable raw size of 1.1+ GB on disk. This ensures us to have enough memory room for subsequent analysis and visualization without crashing our Colab session.



### 2.1. Taxi Trip Records (June and July)

In [ ]:
# Download data using Taxi object (see module taxi.py)
taxi = TaxiData()
taxi_df = taxi.download_data()

>> All files are saved to local disk.


In [ ]:
# Read saved datasets into pandas DataFrame
yellow_taxi_path = lambda m: f"yellow_tripdata_2025-{m}.parquet"
june_df = pd.read_parquet(yellow_taxi_path("06"))
july_df = pd.read_parquet(yellow_taxi_path("07"))

# Merge the 2 datasets into a master dataset
taxi_df = pd.concat([june_df, july_df], ignore_index=True)

# Free up memory
del june_df
del july_df
gc.collect()

# Display the first 10 rows
taxi_df.head(10)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-06-01 00:02:50,2025-06-01 00:39:51,1.0,10.00,1.0,N,138,50,1,47.8,11.00,0.5,20.15,6.94,1.0,87.39,2.5,1.75,0.75
1,2,2025-06-01 00:11:27,2025-06-01 00:35:35,1.0,3.93,1.0,N,158,237,1,24.7,1.00,0.5,6.09,0.00,1.0,36.54,2.5,0.00,0.75
2,1,2025-06-01 00:43:47,2025-06-01 00:49:16,0.0,0.70,1.0,N,230,163,1,7.2,4.25,0.5,2.59,0.00,1.0,15.54,2.5,0.00,0.75
3,1,2025-06-01 00:01:15,2025-06-01 00:42:16,1.0,17.00,2.0,N,132,232,1,70.0,3.25,0.5,5.00,0.00,1.0,79.75,2.5,0.00,0.75
4,7,2025-06-01 00:16:32,2025-06-01 00:16:32,1.0,2.22,1.0,N,48,234,1,20.5,0.00,0.5,5.25,0.00,1.0,31.50,2.5,0.00,0.75
5,1,2025-06-01 00:05:23,2025-06-01 00:16:57,0.0,0.90,1.0,N,164,90,2,11.4,4.25,0.5,0.00,0.00,1.0,17.15,2.5,0.00,0.75
6,1,2025-06-01 00:23:04,2025-06-01 00:35:25,0.0,1.90,1.0,N,246,113,1,12.8,4.25,0.5,3.70,0.00,1.0,22.25,2.5,0.00,0.75
7,1,2025-06-01 00:37:37,2025-06-01 00:42:28,0.0,0.70,1.0,N,113,113,1,7.2,4.25,0.5,2.55,0.00,1.0,15.50,2.5,0.00,0.75
8,1,2025-06-01 00:44:28,2025-06-01 00:50:01,0.0,0.50,1.0,N,249,249,1,7.2,4.25,0.5,2.55,0.00,1.0,15.50,2.5,0.00,0.75
9,1,2025-06-01 00:52:28,2025-06-01 01:03:29,1.0,2.50,1.0,N,249,142,1,13.5,4.25,0.5,5.75,0.00,1.0,25.00,2.5,0.00,0.75


### 2.2. Taxi Zone Lookup Table

The [NYC TLC](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page) website also provides a lookup table for the taxi zones to match with `PULocationID` and `DOLocationID`. The following code downloads the CSV file and puts into a pandas DataFrame.

In [ ]:
# Download the lookup table
taxi.download_taxi_zones()
zones_df = pd.read_csv('taxi_zone_lookup.csv')
zones_df

>> Taxi Lookup Table successfully downloaded


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone
...,...,...,...,...
260,261,Manhattan,World Trade Center,Yellow Zone
261,262,Manhattan,Yorkville East,Yellow Zone
262,263,Manhattan,Yorkville West,Yellow Zone
263,264,Unknown,NaN,NaN


The `Borough` consists of the 6 boroughs of New York city.

In [ ]:
zones_df['Borough'].unique()

array(['EWR', 'Queens', 'Bronx', 'Manhattan', 'Staten Island', 'Brooklyn',
       'Unknown', nan], dtype=object)

Now, let's merge the lookup table with the `taxi_df`.

In [ ]:
# Merge on the Pickup Location ID - use only 'Borough' column
taxi_df = taxi_df.merge(
    zones_df[['LocationID', 'Borough']].rename(columns={'Borough': 'pickup_borough'}),
    left_on='PULocationID',
    right_on='LocationID',
    how='left'
).drop(columns=['PULocationID','LocationID'])


# Merge on the Dropoff Location ID - use only 'Borough' column
taxi_df = taxi_df.merge(
    zones_df[['LocationID', 'Borough']].rename(columns={'Borough': 'dropoff_borough'}),
    left_on='DOLocationID',
    right_on='LocationID',
    how='left'
).drop(columns=['DOLocationID','LocationID'])


# Convert dtype to a `category` type
taxi_df['pickup_borough'] = taxi_df['pickup_borough'].astype('category')
taxi_df['dropoff_borough'] = taxi_df['dropoff_borough'].astype('category')


# Free up memory
del zones_df
del yellow_taxi_path
%reset -f out
gc.collect()

Flushing output cache (3 entries)


0

The following shows the summary of the `taxi_df`. The `info()` confirms that our merged dataset has the **uncompressed size of 1.1+GB**.

In [ ]:
taxi_df.info(memory_usage=True)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8221923 entries, 0 to 8221922
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   payment_type           int64         
 8   fare_amount            float64       
 9   extra                  float64       
 10  mta_tax                float64       
 11  tip_amount             float64       
 12  tolls_amount           float64       
 13  improvement_surcharge  float64       
 14  total_amount           float64       
 15  congestion_surcharge   float64       
 16  Airport_fee            float64       
 17  cbd_congestion_fee     float64       
 18  pickup_borough        

Equivalently, the formula below shows the (uncompressed) memory usage of our dataset is **1.35 GB**.

In [ ]:
(taxi_df.memory_usage(deep=True).sum() / 1024**3).item()

1.3544141557067633

### 2.3. Yellow Trips Data Dictionary
The [Yellow Trips Data Dictionary](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf) describes the meaning of each column:
- **Identifier Feature:** `VendorID`
- **Time Features:** `tpep_pickup_datetime`, `tpep_dropoff_datetime`
- **Geographical Features:** `PULocationID`, `DOLocationID`, `trip_distance`
- **Categorical Features:** `RatecodeID`, `payment_type`, `passenger_count`
- **Money-value Features:** `fare_amount`, `extra`, `mta_tax`, `tip_amount`, `tolls_amount`, `improvement_surcharge`, `total_amount`, `congestion_surcharge`, `airport_fee`, `cbd_congestion_fee`
- **Others:** `store_and_fwd_flag`



In [ ]:
# Show the master dataset
taxi_df.head(5)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,pickup_borough,dropoff_borough
0,1,2025-06-01 00:02:50,2025-06-01 00:39:51,1.0,10.00,1.0,N,1,47.8,11.00,0.5,20.15,6.94,1.0,87.39,2.5,1.75,0.75,Queens,Manhattan
1,2,2025-06-01 00:11:27,2025-06-01 00:35:35,1.0,3.93,1.0,N,1,24.7,1.00,0.5,6.09,0.00,1.0,36.54,2.5,0.00,0.75,Manhattan,Manhattan
2,1,2025-06-01 00:43:47,2025-06-01 00:49:16,0.0,0.70,1.0,N,1,7.2,4.25,0.5,2.59,0.00,1.0,15.54,2.5,0.00,0.75,Manhattan,Manhattan
3,1,2025-06-01 00:01:15,2025-06-01 00:42:16,1.0,17.00,2.0,N,1,70.0,3.25,0.5,5.00,0.00,1.0,79.75,2.5,0.00,0.75,Queens,Manhattan
4,7,2025-06-01 00:16:32,2025-06-01 00:16:32,1.0,2.22,1.0,N,1,20.5,0.00,0.5,5.25,0.00,1.0,31.50,2.5,0.00,0.75,Manhattan,Manhattan


## 3.Data Wrangling

The first step of data wrangling is to remove unrelevant columns, such as `store_and_fwd_flag`.

In [ ]:
taxi_df = taxi_df.drop(columns=['store_and_fwd_flag'])

### 3.1. Handling Duplicates

The next step is to handling duplicated entries. The dataset has 4 duplicates.

In [ ]:
# Compute the number of duplicates
cols = ['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime', 'trip_distance', 'total_amount']
num_duplicates = taxi_df.duplicated(subset=cols).sum()
int(num_duplicates)

4

Let's inspect the duplicated rows.

In [ ]:
# Retrieve the duplicated entries
duplicates = taxi_df[taxi_df.duplicated(subset=cols, keep=False)]

# Sort by pickup time to inspect the duplicates
duplicates.sort_values(by='tpep_pickup_datetime').head(10)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,pickup_borough,dropoff_borough
2296472,1,2025-06-23 07:27:34,2025-06-23 07:27:43,1.0,0.00,1.0,4,3.0,0.00,0.5,0.00,0.0,1.0,4.50,0.0,0.00,0.00,Unknown,Unknown
2297545,1,2025-06-23 07:27:34,2025-06-23 07:27:43,1.0,0.00,1.0,3,3.0,0.00,0.5,0.00,0.0,1.0,4.50,0.0,0.00,0.00,Unknown,Unknown
2399557,1,2025-06-24 08:46:13,2025-06-24 09:00:08,0.0,1.30,1.0,1,13.5,3.25,0.5,3.65,0.0,1.0,21.90,2.5,0.00,0.75,Manhattan,Manhattan
2402374,1,2025-06-24 08:46:13,2025-06-24 09:00:08,1.0,1.30,1.0,1,13.5,3.25,0.5,3.65,0.0,1.0,21.90,2.5,0.00,0.75,Manhattan,Manhattan
4681695,2,2025-07-05 17:19:29,2025-07-05 17:19:34,1.0,0.00,1.0,2,3.0,0.00,0.5,0.00,0.0,1.0,6.25,0.0,1.75,0.00,Queens,Queens
4681810,2,2025-07-05 17:19:29,2025-07-05 17:19:34,1.0,0.00,1.0,2,3.0,0.00,0.5,0.00,0.0,1.0,6.25,0.0,1.75,0.00,Queens,Queens
6284039,2,2025-07-22 19:56:50,2025-07-22 20:06:40,3.0,0.82,1.0,1,10.0,2.50,0.5,0.00,0.0,1.0,17.25,2.5,0.00,0.75,Manhattan,Manhattan
6287100,2,2025-07-22 19:56:50,2025-07-22 20:06:40,1.0,0.82,1.0,4,10.0,2.50,0.5,0.00,0.0,1.0,17.25,2.5,0.00,0.75,Manhattan,Manhattan


The duplicated rows are almost identical with the same `VendorID`, pick up times and drop off times despite of the few disparencies in few columns. These represent system duplicates with wrong data entries because a customer cannot take 2 different rides at the same time. Thus, we removed the 4 duplicates.

In [ ]:
# Drop the duplicated row
taxi_df = taxi_df.drop_duplicates(subset=cols, keep='first')

# Free up memory
del num_duplicates
del duplicates
%reset -f out
gc.collect()

Flushing output cache (5 entries)


0

**Mirrored Entries**

We also found 150,444 mirrorred rows in the dataset.

In [ ]:
# Compute the number of mirrored rows
cols = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'trip_distance']
nb_mirrored = taxi_df.duplicated(subset=cols).sum()
int(nb_mirrored)

150444

In [ ]:
# Retrieve the duplicated/mirrored entries
duplicates = taxi_df[taxi_df.duplicated(subset=cols, keep=False)]

# Sort by pickup time to inspect the duplicates/mirrored
duplicates.sort_values(by='tpep_pickup_datetime').head(10)

,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,pickup_borough,dropoff_borough
3339,2,2025-06-01 00:00:30,2025-06-01 00:20:55,1.0,3.63,1.0,4,21.9,1.0,0.5,0.0,0.0,1.0,27.65,2.5,0.00,0.75,Manhattan,Brooklyn
3338,2,2025-06-01 00:00:30,2025-06-01 00:20:55,1.0,3.63,1.0,4,-21.9,-1.0,-0.5,0.0,0.0,-1.0,-27.65,-2.5,0.00,-0.75,Manhattan,Brooklyn
4562,2,2025-06-01 00:00:38,2025-06-01 00:08:35,2.0,1.51,1.0,2,9.3,1.0,0.5,0.0,0.0,1.0,15.05,2.5,0.00,0.75,Manhattan,Manhattan
4561,2,2025-06-01 00:00:38,2025-06-01 00:08:35,2.0,1.51,1.0,2,-9.3,-1.0,-0.5,0.0,0.0,-1.0,-15.05,-2.5,0.00,-0.75,Manhattan,Manhattan
4476,2,2025-06-01 00:00:42,2025-06-01 00:16:44,1.0,11.23,1.0,3,42.2,1.0,0.5,0.0,0.0,1.0,46.45,0.0,1.75,0.00,Queens,Queens
4475,2,2025-06-01 00:00:42,2025-06-01 00:16:44,1.0,11.23,1.0,3,-42.2,-1.0,-0.5,0.0,0.0,-1.0,-46.45,0.0,-1.75,0.00,Queens,Queens
2078,2,2025-06-01 00:00:52,2025-06-01 00:01:42,1.0,0.10,1.0,4,-3.0,-1.0,-0.5,0.0,0.0,-1.0,-8.75,-2.5,0.00,-0.75,Manhattan,Manhattan
2079,2,2025-06-01 00:00:52,2025-06-01 00:01:42,1.0,0.10,1.0,4,3.0,1.0,0.5,0.0,0.0,1.0,8.75,2.5,0.00,0.75,Manhattan,Manhattan
4811,2,2025-06-01 00:00:54,2025-06-01 00:00:59,1.0,0.00,1.0,3,-3.0,-1.0,-0.5,0.0,0.0,-1.0,-5.50,0.0,0.00,0.00,Queens,Queens
4812,2,2025-06-01 00:00:54,2025-06-01 00:00:59,1.0,0.00,1.0,3,3.0,1.0,0.5,0.0,0.0,1.0,5.50,0.0,0.00,0.00,Queens,Queens


The **mirrored records** are twins to each other.
- One contains a **positive** `total_amount`.
- The other contains an **identical negative value**.

These represent transactions that were accidentally registered and then cancelled by the drivers. We removed these 150,444 mirrored records to prevent statistical inflation caused by the negative financial values (such as total amount, fees, fares, tips, etc) and to ensure a more meaningful analysis of customer behavior.

In [ ]:
# Apply absolute value on the total amount
taxi_df['abs_amount'] = taxi_df['total_amount'].abs()
cols = ['tpep_pickup_datetime', 'tpep_dropoff_datetime', 'trip_distance', 'abs_amount']

# Find the mirrored rows (Positive/Negative pairs)
is_mirror = taxi_df.duplicated(subset=cols, keep=False)
taxi_df[is_mirror][['VendorID', 'trip_distance','total_amount','abs_amount']].head(10)

,VendorID,trip_distance,total_amount,abs_amount
80,2,0.14,-9.45,9.45
81,2,0.14,9.45,9.45
224,2,0.83,-12.95,12.95
225,2,0.83,12.95,12.95
264,2,0.00,-74.00,74.00
265,2,0.00,74.00,74.00
331,2,0.02,-8.75,8.75
332,2,0.02,8.75,8.75
458,2,3.17,-22.70,22.70
459,2,3.17,22.70,22.70


In [ ]:
# Drop mirrored rows
taxi_df = taxi_df[~is_mirror]

### 3.2. Handle 0-distance rows

The `taxi_df` contains 0-distance rows, indicating the taximeter did not record the trip distance in miles. We decided to drop these 238,440 0-distance rows to ensure a meaningful predicitons for the Machine Learning models.

In [ ]:
# Inspect the number of 0-distance rows
print("Number of 0-distance rows:", len(taxi_df.loc[taxi_df["trip_distance"]==0]))

# Drop the 0-distance rows
taxi_df = taxi_df[taxi_df['trip_distance'] > 0]
print("Successfully dropped")

Number of 0-distance rows: 238440
Successfully dropped


**Check**: There is no 0-distance rows.

In [ ]:
0 in taxi_df['trip_distance'].unique()

False

### 3.3. Handle 0-passenger rows

Similarly, The `taxi_df` contains 0-passenger rows. We decided to replace these 0-passenger rows (0.54% of the dataset) with the mode to ensure a meaningful predicitons for the Machine Learning models. A trip should have at least 1 passenger.

In [ ]:
zero_passenger_df = taxi_df.loc[taxi_df['passenger_count']==0]
print(f"Number of 0-passenger rows: {len(zero_passenger_df)}")
print(f"Ratio of 0-passenger rows: {len(zero_passenger_df)/len(taxi_df)*100: .2f} %")

Number of 0-passenger rows: 41675
Ratio of 0-passenger rows:  0.54 %


The following indicates that the 0-passenger rows do not have `RatecodeID` 6 (Group ride). Thus, we can safely replace 0 with the mode (which is 1 passenger).

In [ ]:
# Inspect the unique values and mode
print("Unique values:", zero_passenger_df['RatecodeID'].unique())
print("Mode:", zero_passenger_df['RatecodeID'].mode().item())

# Replace 0 passenger with mode 1
taxi_df.loc[taxi_df['passenger_count'] == 0, 'passenger_count'] = 1
print("Successfully replace 0 passenger")

Unique values: [ 1.  2.  3.  5. 99.  4.]
Mode: 1.0
Successfully replace 0 passenger


**Check:** There is no 0-passenger rows.

In [ ]:
taxi_df['passenger_count'].unique()

array([ 1.,  2.,  3.,  4.,  5.,  6.,  9.,  8.,  7., nan])

### 3.4. Handle NaN values

In [ ]:
# Check the number of NaN values in each column
taxi_df.isnull().sum()

,0
VendorID,0
tpep_pickup_datetime,0
tpep_dropoff_datetime,0
passenger_count,2074024
trip_distance,0
RatecodeID,2074024
payment_type,0
fare_amount,0
extra,0
mta_tax,0


**PART 1**

The ratio of NaN values counts for 27%, and the number of NaN is consistent accross these 4 features:
- `passenger_count`
- `RatecodeID`
- `congestion_surcharge`
- `Airport_fee`

In [ ]:
# Compute ratio of NaN values
nan_count = taxi_df['congestion_surcharge'].isnull().sum()
nan_ratio = round(nan_count/len(taxi_df) * 100)
print(f"Ratio of NaN values: {nan_ratio} %")

Ratio of NaN values: 27 %


For our analysis, we decided to **impute the missing values** in `congestion_surcharge` and `Airport_fee` with 0. These fees are redundant as they are already captured by the `RatecodeID` feature:

- `RatecodeID` 2 (JFK) or 3 (Newark) indicates an airport trip with an additional fee between `1.75$` or `6.75$`.
- `RatecodeID` 1 (Standard rate) is used for trips within the Manhattan congestion zone (with a surcharge) and other boroughs (without a surcharge).

Replacing NaNs by 0 allows us to represent that a passenger did not travel to the airport or enter the Manhanttan congestion zone.


In [ ]:
print('Airport_fee')
print('---------------------')
print('Unique values:', taxi_df['Airport_fee'].unique())
print('Average', taxi_df.loc[taxi_df['Airport_fee'] > 0, 'Airport_fee'].mean().round(2))

print('\n\nCongestion Surcharge')
print('---------------------')
print('Unique values:', taxi_df['congestion_surcharge'].unique())
print('Average', taxi_df.loc[taxi_df['congestion_surcharge'] > 0, 'congestion_surcharge'].mean().round(2))

Airport_fee
---------------------
Unique values: [ 1.75  0.    6.75  5.   -1.75   nan]
Average 1.79


Congestion Surcharge
---------------------
Unique values: [ 2.5   0.   -2.5    nan  0.75  1.  ]
Average 2.5


In [ ]:
# Replace NaNs with 0
taxi_df.loc[:, ['congestion_surcharge', 'Airport_fee']] = taxi_df[['congestion_surcharge', 'Airport_fee']].fillna(0)

Using a similar logic, we imputed `passenger_count` with its mode, which is 1. This allows us to represent a single-passenger trip, which is the most frequence occurence in our dataset. For `RatecodeID`, we use the code 99 to fill the missing values, as that code represents an Unknown rate.

In [ ]:
# Show the modes
print('Mode for passenger_count:', taxi_df['passenger_count'].mode().item())

Mode for RatecodeID: 1.0
Mode for passenger_count: 1.0


In [ ]:
# Replace NaNs with mode 1
taxi_df.loc[:, ['passenger_count']] = taxi_df[['passenger_count']].fillna(1)
# Replace NaNs with code 99
taxi_df.loc[:, ['RatecodeID']] = taxi_df[['RatecodeID']].fillna(99)

**PART 2**

The dataset contains NaN in the `pickup_borough` and `dropoff_borough` columns. We decided to replace these missing values with **'Unknown'**, which is an existing category in the lookup table. This allows us to have a consistent and clear geographic data accross the dataset without the need to remove these records.

In [ ]:
# Replace NaNs with 'Unknown'
taxi_df.loc[taxi_df['pickup_borough'].isnull(), 'pickup_borough'] = 'Unknown'
taxi_df.loc[taxi_df['dropoff_borough'].isnull(), 'dropoff_borough'] = 'Unknown'

**Check:** The dataset has no more missing values.

In [ ]:
taxi_df.isnull().sum()

,0
VendorID,0
tpep_pickup_datetime,0
tpep_dropoff_datetime,0
passenger_count,0
trip_distance,0
RatecodeID,0
payment_type,0
fare_amount,0
extra,0
mta_tax,0


### 3.5. Label Mapping

This section consists of converting numerical codes (1,2,3, etc) to their corresponding descriptive labels. We perform label mapping in the following features by using the [Yellow Trips Data Dictionary](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf).


- `RatecodeID`
- `payment_type`

#### 3.5.1 Feature: `RatecodeID`

In [ ]:
# Show the unique numerical codes
taxi_df['RatecodeID'].unique()

array([ 1.,  2.,  5., 99.,  4.,  3.,  6.])

In [ ]:
# Define the rate code dictionary
rate_code_map = {
    1: "Standard rate",
    2: "JFK",
    3: "Newark",
    4: "Nassau or Westchester",
    5: "Negotiated fare",
    6: "Group ride",
    99: "Unknown"
}

# Map the rate code ID with the appropriate category
taxi_df.loc[:,'rate_code'] = taxi_df['RatecodeID'].map(rate_code_map).astype('category')
taxi_df = taxi_df.drop(columns=['RatecodeID'])

# Free up memory
%reset -f out
gc.collect()

Flushing output cache (9 entries)


0

**Check:** The rate codes are correctly mapped.

In [ ]:
taxi_df['rate_code'].unique()

['Standard rate', 'JFK', 'Negotiated fare', 'Unknown', 'Nassau or Westchester', 'Newark', 'Group ride']
Categories (7, object): ['Group ride', 'JFK', 'Nassau or Westchester', 'Negotiated fare', 'Newark',
                         'Standard rate', 'Unknown']

#### 3.5.2 Feature: `payment_type`

In [ ]:
# Show the unique numerical codes
taxi_df['payment_type'].unique()

array([1, 2, 3, 4, 5, 0])

In [ ]:
# Define the payment dictionary
payment_map = {
    0: "Flex Fare",
    1: "Credit card",
    2: "Cash",
    3: "No charge",
    4: "Dispute",
    5: "Unknown",
    6: "Voided trip"
}

# Apply mapping and categorize
taxi_df.loc[:,'payment_type']= taxi_df['payment_type'].map(payment_map).astype('category')

/tmp/ipython-input-103069/2337347168.py:13: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '['Credit card', 'Credit card', 'Credit card', 'Credit card', 'Credit card', ..., 'Flex Fare', 'Flex Fare', 'Flex Fare', 'Flex Fare', 'Flex Fare']
Length: 7683649
Categories (6, object): ['Cash', 'Credit card', 'Dispute', 'Flex Fare', 'No charge', 'Unknown']' has dtype incompatible with int64, please explicitly cast to a compatible dtype first.
  taxi_df.loc[:,'payment_type']= taxi_df['payment_type'].map(payment_map).astype('category')


**Check:** The payment types are correctly mapped.

In [ ]:
taxi_df['payment_type'].unique()

['Credit card', 'Cash', 'No charge', 'Dispute', 'Unknown', 'Flex Fare']
Categories (6, object): ['Cash', 'Credit card', 'Dispute', 'Flex Fare', 'No charge', 'Unknown']

### 3.6. Final clean up

In [ ]:
# Drop unrelevant columns
taxi_df = taxi_df.drop(columns=['VendorID','abs_amount'])

# Delete all temporay variable and Free up memory
del nan_count
del nan_ratio
del nb_mirrored
del is_mirror
del rate_code_map
del payment_map
%reset -f out
gc.collect()

Flushing output cache (4 entries)


0

Let's reduce the numerical dtype to a smaller dtype to efficently manage memory usage.

In [ ]:
# Define the numerical columns to convert
float64_cols = [
    'trip_distance', 'fare_amount', 'extra', 'mta_tax', 'tip_amount',
    'tolls_amount', 'improvement_surcharge', 'total_amount',
    'congestion_surcharge','Airport_fee', 'cbd_congestion_fee'
]

# Convert to a smaller dtype
taxi_df['passenger_count'] = taxi_df['passenger_count'].astype('int8')
taxi_df[float64_cols] = taxi_df[float64_cols].astype('float32')

Here is the updated `taxi_df`.

In [ ]:
taxi_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7683649 entries, 0 to 8221922
Data columns (total 18 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   tpep_pickup_datetime   datetime64[us]
 1   tpep_dropoff_datetime  datetime64[us]
 2   passenger_count        int8          
 3   trip_distance          float32       
 4   payment_type           category      
 5   fare_amount            float32       
 6   extra                  float32       
 7   mta_tax                float32       
 8   tip_amount             float32       
 9   tolls_amount           float32       
 10  improvement_surcharge  float32       
 11  total_amount           float32       
 12  congestion_surcharge   float32       
 13  Airport_fee            float32       
 14  cbd_congestion_fee     float32       
 15  pickup_borough         category      
 16  dropoff_borough        category      
 17  rate_code              category      
dtypes: category(4), datetime64[

In [ ]:
taxi_df.head(5)

,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee,pickup_borough,dropoff_borough,rate_code
0,2025-06-01 00:02:50,2025-06-01 00:39:51,1,10.00,Credit card,47.799999,11.00,0.5,20.15,6.94,1.0,87.389999,2.5,1.75,0.75,Queens,Manhattan,Standard rate
1,2025-06-01 00:11:27,2025-06-01 00:35:35,1,3.93,Credit card,24.700001,1.00,0.5,6.09,0.00,1.0,36.540001,2.5,0.00,0.75,Manhattan,Manhattan,Standard rate
2,2025-06-01 00:43:47,2025-06-01 00:49:16,1,0.70,Credit card,7.200000,4.25,0.5,2.59,0.00,1.0,15.540000,2.5,0.00,0.75,Manhattan,Manhattan,Standard rate
3,2025-06-01 00:01:15,2025-06-01 00:42:16,1,17.00,Credit card,70.000000,3.25,0.5,5.00,0.00,1.0,79.750000,2.5,0.00,0.75,Queens,Manhattan,JFK
4,2025-06-01 00:16:32,2025-06-01 00:16:32,1,2.22,Credit card,20.500000,0.00,0.5,5.25,0.00,1.0,31.500000,2.5,0.00,0.75,Manhattan,Manhattan,Standard rate


### 3.7. `quantDDA()` and `vizDDA()`

In [ ]:
dda = TaxiDDA()           # Using TaxiDDA() from taxi_utils.py
dda.quantDDA(taxi_df)

,feature,num_observations,num_entries,num_unique,num_missing,num_outlier,num_extreme,mode,mean,std,min,Q1,median,Q3,max,skew,kurtosis
0,tpep_pickup_datetime,7683649,7683649,3605214,0,nan,nan,2025-06-05T19:27:00.000000,nan,nan,nan,nan,nan,nan,nan,nan,nan
1,tpep_dropoff_datetime,7683649,7683649,3605581,0,nan,nan,2025-06-05T20:02:00.000000,nan,nan,nan,nan,nan,nan,nan,nan,nan
2,passenger_count,7683649,7683649,9,0,1128423,43104,1,1.22,0.64,1,1.0,1.0,1.0,9,3.648,15.498
3,trip_distance,7683649,7683649,5978,0,863854,149221,0.9,7.66,706.33,0.009999999776482582,1.15,2.0,4.0,397994.375,219.599,60052.895
4,payment_type,7683649,7683649,6,0,nan,nan,Credit card,nan,nan,nan,nan,nan,nan,nan,nan,nan
5,fare_amount,7683649,7683649,11810,0,594490,90493,8.6,19.41,117.97,-483.20001220703125,9.3,14.2,23.45,325478.0625,2677.608,7336971.5
6,extra,7683649,7683649,78,0,92930,49984,0.0,1.19,1.82,-7.5,0.0,0.0,2.5,42.459999084472656,1.795,3.611
7,mta_tax,7683649,7683649,64,0,70901,70901,0.5,0.5,1.89,-21.739999771118164,0.5,0.5,0.5,5243.3798828125,2768.559,7671157.0
8,tip_amount,7683649,7683649,5284,0,413817,73427,0.0,2.87,3.94,-8.59000015258789,0.0,2.0,4.0,960.9400024414062,5.96,546.24
9,tolls_amount,7683649,7683649,1418,0,541850,35620,0.0,0.53,2.07,-70.0,0.0,0.0,0.0,716.0499877929688,10.415,1782.017


In [ ]:
# Sample a few rows for vizDDA()
# samples_df
# dda.vizDDA(samples_df)

In [ ]:
# Free up memory
plt.close('all')
gc.collect()

## 4. Exploratory Data Analysis (EDA)

### 4.1 Summary Statistics

In [ ]:
import seaborn as sns
import numpy as np

sns.set_theme(style='whitegrid', palette='muted')

# ── Numerical summary ─────────────────────────────────────────────────────
numerical_cols = [
    'trip_distance', 'fare_amount', 'extra', 'mta_tax', 'tip_amount',
    'tolls_amount', 'improvement_surcharge', 'total_amount',
    'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee', 'passenger_count'
]

print(f"Dataset shape: {taxi_df.shape[0]:,} rows × {taxi_df.shape[1]} columns\n")
taxi_df[numerical_cols].describe().round(2)

In [ ]:
# ── Categorical summary ───────────────────────────────────────────────────
categorical_cols = ['payment_type', 'rate_code', 'pickup_borough', 'dropoff_borough']

for col in categorical_cols:
    vc  = taxi_df[col].value_counts()
    pct = taxi_df[col].value_counts(normalize=True).mul(100).round(2)
    summary = pd.DataFrame({'Count': vc, 'Percentage (%)': pct})
    print(f"\n{'='*50}\n  {col}\n{'='*50}")
    print(summary.to_string())

### 4.2 Univariate Visualizations

Distributions of key trip and financial features. A 50 000-row random sample is used for plotting efficiency; upper bounds are clipped to improve readability while covering the vast majority of trips.

In [ ]:
# ── Time features used throughout EDA ─────────────────────────────────────
taxi_df['pickup_hour'] = taxi_df['tpep_pickup_datetime'].dt.hour
taxi_df['pickup_day']  = taxi_df['tpep_pickup_datetime'].dt.day_name()

# 50 k random sample for plots (memory-safe)
sample_df = taxi_df.sample(n=50_000, random_state=42)

# ── 3 × 3 univariate figure ────────────────────────────────────────────────
fig, axes = plt.subplots(3, 3, figsize=(16, 13))
fig.suptitle('Univariate Distributions — Key Features', fontsize=15, fontweight='bold')

# 1. Trip Distance
axes[0,0].hist(sample_df['trip_distance'].clip(0, 20), bins=60, color='steelblue', edgecolor='white')
axes[0,0].set_title('Trip Distance (miles)'); axes[0,0].set_xlabel('Miles'); axes[0,0].set_ylabel('Count')

# 2. Fare Amount
axes[0,1].hist(sample_df['fare_amount'].clip(0, 80), bins=60, color='seagreen', edgecolor='white')
axes[0,1].set_title('Fare Amount ($)'); axes[0,1].set_xlabel('$'); axes[0,1].set_ylabel('Count')

# 3. Tip Amount
axes[0,2].hist(sample_df['tip_amount'].clip(0, 25), bins=60, color='mediumpurple', edgecolor='white')
axes[0,2].set_title('Tip Amount ($)'); axes[0,2].set_xlabel('$'); axes[0,2].set_ylabel('Count')

# 4. Total Amount
axes[1,0].hist(sample_df['total_amount'].clip(0, 100), bins=60, color='coral', edgecolor='white')
axes[1,0].set_title('Total Amount ($)'); axes[1,0].set_xlabel('$'); axes[1,0].set_ylabel('Count')

# 5. Passenger Count
pc = taxi_df['passenger_count'].value_counts().sort_index()
axes[1,1].bar(pc.index, pc.values, color='darkorange', edgecolor='white')
axes[1,1].set_title('Passenger Count'); axes[1,1].set_xlabel('Passengers'); axes[1,1].set_ylabel('Count')
axes[1,1].set_xticks(pc.index)

# 6. Pickup Hour
hr = taxi_df['pickup_hour'].value_counts().sort_index()
axes[1,2].bar(hr.index, hr.values, color='teal', edgecolor='white')
axes[1,2].set_title('Pickup Hour of Day'); axes[1,2].set_xlabel('Hour'); axes[1,2].set_ylabel('Count')

# 7. Payment Type
pt = taxi_df['payment_type'].value_counts()
axes[2,0].barh(pt.index, pt.values, color='royalblue', edgecolor='white')
axes[2,0].set_title('Payment Type'); axes[2,0].set_xlabel('Count')

# 8. Pickup Borough
pb = taxi_df['pickup_borough'].value_counts()
axes[2,1].barh(pb.index, pb.values, color='tomato', edgecolor='white')
axes[2,1].set_title('Pickup Borough'); axes[2,1].set_xlabel('Count')

# 9. Rate Code
rc = taxi_df['rate_code'].value_counts()
axes[2,2].barh(rc.index, rc.values, color='goldenrod', edgecolor='white')
axes[2,2].set_title('Rate Code'); axes[2,2].set_xlabel('Count')

plt.tight_layout()
plt.savefig('eda_univariate.png', dpi=120, bbox_inches='tight')
plt.show()

### 4.3 Bivariate Visualizations

Relationships between trip features and financial outcomes, and how demand varies by time and geography.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.suptitle('Bivariate Analysis', fontsize=15, fontweight='bold')

# 1. Fare Amount vs Trip Distance
axes[0,0].scatter(sample_df['trip_distance'].clip(0, 30),
                  sample_df['fare_amount'].clip(0, 100),
                  alpha=0.04, s=5, color='steelblue')
axes[0,0].set_title('Fare Amount vs Trip Distance')
axes[0,0].set_xlabel('Trip Distance (miles)'); axes[0,0].set_ylabel('Fare ($)')

# 2. Tip Amount by Payment Type (box plot highlights cash = no tips)
tip_cc   = sample_df.loc[sample_df['payment_type'] == 'Credit card', 'tip_amount'].clip(0, 25)
tip_cash = sample_df.loc[sample_df['payment_type'] == 'Cash',        'tip_amount'].clip(0, 25)
axes[0,1].boxplot([tip_cc, tip_cash], labels=['Credit card', 'Cash'],
                  patch_artist=True, medianprops=dict(color='black', linewidth=2),
                  boxprops=dict(facecolor='steelblue', alpha=0.6))
axes[0,1].set_title('Tip Amount by Payment Type')
axes[0,1].set_xlabel('Payment Type'); axes[0,1].set_ylabel('Tip ($)')

# 3. Total Amount by Pickup Borough
boroughs = [b for b in taxi_df['pickup_borough'].cat.categories if b != 'Unknown']
borough_order = (taxi_df[taxi_df['pickup_borough'].isin(boroughs)]
                 .groupby('pickup_borough')['total_amount'].median()
                 .sort_values(ascending=False).index.tolist())
sns.boxplot(data=sample_df[sample_df['pickup_borough'].isin(borough_order)],
            x='pickup_borough', y='total_amount', order=borough_order,
            ax=axes[0,2], palette='Set2')
axes[0,2].set_title('Total Amount by Pickup Borough')
axes[0,2].set_xlabel('Borough'); axes[0,2].set_ylabel('Total ($)')
axes[0,2].set_ylim(0, 80); axes[0,2].tick_params(axis='x', rotation=20)

# 4. Average Fare by Hour of Day
hourly_fare = taxi_df.groupby('pickup_hour')['fare_amount'].mean()
axes[1,0].plot(hourly_fare.index, hourly_fare.values, marker='o', color='coral', linewidth=2)
axes[1,0].set_title('Avg Fare by Hour of Day')
axes[1,0].set_xlabel('Hour'); axes[1,0].set_ylabel('Avg Fare ($)')
axes[1,0].set_xticks(range(0, 24, 2))

# 5. Average Tip by Hour of Day
hourly_tip = taxi_df.groupby('pickup_hour')['tip_amount'].mean()
axes[1,1].plot(hourly_tip.index, hourly_tip.values, marker='o', color='mediumpurple', linewidth=2)
axes[1,1].set_title('Avg Tip by Hour of Day')
axes[1,1].set_xlabel('Hour'); axes[1,1].set_ylabel('Avg Tip ($)')
axes[1,1].set_xticks(range(0, 24, 2))

# 6. Trip Count by Day of Week
day_order  = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
day_counts = taxi_df['pickup_day'].value_counts().reindex(day_order)
axes[1,2].bar(day_counts.index, day_counts.values, color='darkorange', edgecolor='white')
axes[1,2].set_title('Trip Count by Day of Week')
axes[1,2].set_xlabel('Day'); axes[1,2].set_ylabel('Trips')
axes[1,2].tick_params(axis='x', rotation=30)

plt.tight_layout()
plt.savefig('eda_bivariate.png', dpi=120, bbox_inches='tight')
plt.show()

### 4.4 Correlation Analysis

Pearson correlation matrix across all numerical features to identify strong linear relationships.

In [ ]:
corr_cols = [
    'trip_distance', 'fare_amount', 'extra', 'mta_tax', 'tip_amount',
    'tolls_amount', 'improvement_surcharge', 'total_amount',
    'congestion_surcharge', 'Airport_fee', 'cbd_congestion_fee',
    'passenger_count', 'pickup_hour'
]

corr_matrix = sample_df[corr_cols].astype('float64').corr()

# ── Top 10 strongest pairs ─────────────────────────────────────────────────
pairs = (corr_matrix
         .where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
         .stack().reset_index())
pairs.columns = ['Feature A', 'Feature B', 'Pearson r']
top10 = pairs.reindex(pairs['Pearson r'].abs().sort_values(ascending=False).index).head(10)
print("Top 10 strongest correlations:\n")
print(top10.to_string(index=False))

# ── Heat-map ───────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f',
            cmap='coolwarm', center=0, square=True, linewidths=0.4,
            annot_kws={'size': 8}, ax=ax)
ax.set_title('Pearson Correlation Matrix — Numerical Features',
             fontsize=14, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('eda_correlation.png', dpi=120, bbox_inches='tight')
plt.show()

### 4.5 Research Questions

---

#### RQ1 — Supervised (Regression)

> **Can we accurately predict the tip amount (`tip_amount`) of a NYC Yellow Taxi trip from trip-level features?**

**Motivation:** The correlation analysis and bivariate plots show that `tip_amount` is strongly correlated with `fare_amount`, `trip_distance`, and `total_amount`, and varies significantly by payment type and pickup borough. A regression model (e.g. Ridge Regression, Gradient Boosted Trees) trained on features such as `trip_distance`, `fare_amount`, `pickup_borough`, `dropoff_borough`, `rate_code`, `passenger_count`, `pickup_hour`, and `pickup_day` could predict tip behaviour. This is practically valuable for drivers estimating expected earnings and for understanding what factors drive tipping.

**Target variable:** `tip_amount` (continuous)  
**Task type:** Supervised — Regression

---

#### RQ2 — Unsupervised (Clustering)

> **Can we identify distinct trip-pattern segments among NYC Yellow Taxi rides during the Peak Summer 2025 season?**

**Motivation:** The univariate and bivariate analyses reveal wide variation in distance, fare, hour of day, and borough. Clustering (e.g. K-Means or DBSCAN) on features such as `trip_distance`, `fare_amount`, `tip_amount`, `pickup_hour`, and encoded borough/payment-type information could surface natural traveller segments — for example, short airport transfers, long late-night rides, or frequent short Manhattan hops. These segments can inform pricing strategy and fleet allocation.

**Features:** `trip_distance`, `fare_amount`, `tip_amount`, `pickup_hour`, encoded `pickup_borough`, `payment_type`  
**Task type:** Unsupervised — Clustering

ToDO
1. Outliers (see quantDDA())
2. Feature Selection
3. Feature Engineering
4. Handle negative fare amounts, total amounts...etc negative fees